# Stage 4.02 — freeze literal manifest and delay-pair reset identities
This freezes exactly 64 rows and audits the 32 task × scene × seed reset identities. No outcome data are read.

In [ ]:
import csv,os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"; PY=Path.home()/"venv-stage4-openvla/bin/python"; OUT=Path.home()/"stage4"; MAN=OUT/"stage4_second_policy_manifest.csv"; AUDIT=OUT/"stage4_initialization_pairing_audit.csv"
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(); plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.make_stage4_manifest","--output",str(MAN),"--git-sha",bench,"--libero-plus-git-sha",plus],cwd=R,check=True)
base=os.environ.copy(); base.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(R)})
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage4.yaml"),"--manifest",str(MAN),"--scene","id","--expected-rows","64","--expected-cells-per-key","2","--audit-output",str(AUDIT)],cwd=R,env=base,check=True)
ood=base.copy(); ood.update({"PYTHONPATH":str(P)+os.pathsep+str(R),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+ood.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+ood.get("LD_LIBRARY_PATH","")})
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage4.yaml"),"--manifest",str(MAN),"--scene","ood","--expected-rows","64","--expected-cells-per-key","2","--audit-output",str(AUDIT)],cwd=R,env=ood,check=True)
rows=list(csv.DictReader(open(MAN))); audit=list(csv.DictReader(open(AUDIT)))
assert len(rows)==64 and len({r['run_id'] for r in rows})==64 and len(audit)==32
assert {r['task_key'] for r in rows}=={'spatial_transport','long_stove_moka'} and {r['seed'] for r in rows}=={str(x) for x in range(38,46)}
assert {r['added_delay_ms'] for r in rows}=={'0','200'} and {r['native_chunk_size'] for r in rows}=={'8'} and {r['request_threshold_actions'] for r in rows}=={'4'}
assert all(r['repeatability_pass']=='True' and r['fingerprint']==r['repeat_fingerprint'] for r in audit)
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.validate_stage4","--manifest",str(MAN),"--output-dir",str(OUT),"--allow-incomplete"],cwd=R,env=base,check=True)
print("PASS: frozen 64-row Stage 4 manifest; 32 reset identities paired across Native/+200")
print("STOP HERE: paste both PASS lines before notebook 03")